# Data Cleaning, Feature Engineering, and Preprocessing

In [34]:
##Load Required Libraries
import pandas as pd
import numpy as np

##Preprocessing Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from imblearn.pipeline import Pipeline

##Splitting Pipeline
from sklearn.model_selection import train_test_split

In [35]:
#Load Data
df = pd.read_csv('../Data/clean_df.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1,1,145.0,233.0,1,2,150.0,0,2.3,3,0,6,0
1,67.0,1,4,160.0,286.0,0,2,108.0,1,1.5,2,3,3,2
2,67.0,1,4,120.0,229.0,0,2,129.0,1,2.6,2,2,7,1
3,37.0,1,3,130.0,250.0,0,0,187.0,0,3.5,3,0,3,0
4,41.0,0,2,130.0,204.0,0,2,172.0,0,1.4,1,0,3,0


# 4. Final Data Cleaning & Feature Engineering

In [36]:
#Rename Categories (with names not encoding)
df['sex'] = df['sex'].replace({0: 'Female', 1: 'Male'})

df['cp'] = df['cp'].replace({
    1: 'Typical Angina',
    2: 'Atypical Angina',
    3: 'Non-Anginal Pain',
    4: 'Asymptomatic'
})

df['fbs'] = df['fbs'].replace({
    0: 'False',
    1: 'True'
})

df['restecg'] = df['restecg'].replace({
    0: 'Normal',
    1: 'ST-T Wave Abnormality', 
    2: 'Possible Left Ventricle Hypertrophy'
})

df['exang'] = df['exang'].replace({
    0: 'No',
    1: 'Yes', 
})

df['slope'] = df['slope'].replace({
    1: 'Upsloping', 
    2: 'Flat', 
    3: 'Downsloping'
})


df['thal'] = df['thal'].replace({
    3: 'Normal', 
    6: 'Fixed Defect', 
    7: 'Reversible Defect'
})

df['ca'] = df['ca'].astype(str)

In [37]:
## Age Buckets
df['age_bucket'] = pd.cut(
    df['age'],
    bins = [0,45,60,100],
    labels = ['Young', 'Middle-Aged', 'Senior']
)
df['age_bucket'] = df['age_bucket'].astype(str)

In [38]:
## OldPeak
df['oldpeak_threshold'] = pd.cut(
    df['oldpeak'],
    bins = [-0.1,1.0,2.0,5.0,df['oldpeak'].max()+0.1],
    labels = ['Low', 'Medium', 'High', 'Very High'],
    include_lowest = True
)
df['oldpeak_threshold'] = df['oldpeak_threshold'].astype(str)

In [39]:
#Target mapping
df['target'] = df['target'].replace({
    0:0,
    1:1,
    2:1,
    3:1,
    4:1
})
df['target'] = df['target'].astype(str)

In [40]:
df.to_csv('../Data/df_preprocess.csv', index = False)

In [41]:
df.dtypes

age                  float64
sex                   object
cp                    object
trestbps             float64
chol                 float64
fbs                   object
restecg               object
thalach              float64
exang                 object
oldpeak              float64
slope                 object
ca                    object
thal                  object
target                object
age_bucket            object
oldpeak_threshold     object
dtype: object

# 5. Preprocessing

In [42]:
##Preprocessing Pipeline

#Define numeric features that need to be scaled and categorical features that need OneHotEncoding
numeric_features = ['age','trestbps','chol','thalach','oldpeak']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope','ca','thal','age_bucket','oldpeak_threshold']

#Create a preprocessing transformer variable
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

#Build the pipeline
pipeline = Pipeline(steps = [
    ('preprocess', preprocessor),
])